In [ ]:
import os
from datasets.prepare import prepare_dataset

# List of datasets you want to install
datasets_to_install = ['orgaquant', 'multiorg']

# Define paths
DOWNLOAD_TEMP_DIR = './tmp'
FINAL_DATA_DIR = './data'

# Create directories if they don't exist
os.makedirs(DOWNLOAD_TEMP_DIR, exist_ok=True)
os.makedirs(FINAL_DATA_DIR, exist_ok=True)

for name in datasets_to_install:
    print(f"\n=== Preparing dataset: {name} ===")
    try:
        prepare_dataset(
            dataset_name=name,
            should_download=True,
            download_dir=DOWNLOAD_TEMP_DIR, # Temporary folder for .zip files
            data_dir=FINAL_DATA_DIR        # Final folder for .h5 files
        )
        print(f"Success: {name}.h5 is ready in the {FINAL_DATA_DIR} folder.")
    except Exception as e:
        print(f"Error during preparation of {name}: {e}")
        

In [3]:
import os
from datasets.prepare import prepare_dataset

# 1. Chemin vers votre dossier existant
# Remplacez par le chemin absolu ou relatif vers multiOrg_benchmark
LOCAL_DATASET_PATH = 'multiOrg_benchmark/multiorg_mmdet_coco' 

# 2. Lancement du prétraitement
try:
    print("Starting local preprocessing...")
    prepare_dataset(
        dataset_name='multiorg', # Doit correspondre au nom dans prepare_utils.py
        should_download=False,   # TRÈS IMPORTANT : on ne télécharge pas
        download_dir=LOCAL_DATASET_PATH, # On pointe vers votre dossier existant
        data_dir='./data'        # Où sera sauvegardé le fichier .h5 final
    )
    print("Success! data/multiorg.h5 has been created.")
except Exception as e:
    print(f"Error: {e}")

Starting local preprocessing...


Success! data/multiorg.h5 has been created.


In [5]:
import os
import csv
import h5py
import json
import numpy as np
from PIL import Image
from pathlib import Path
from tqdm import tqdm

def prepare_multiorg_coco(in_dir: str, out_path: str) -> None:
    """Prepare the MultiOrg dataset from COCO format crops to H5."""
    import json
    from PIL import Image

    with h5py.File(out_path, 'w') as hdf:
        # On traite 'train' et 'val' (car ton script a renommé 'test' en 'val')
        for split in ['train', 'val']:
            print(f"Converting {split} to H5...")
            group = hdf.create_group(split if split == 'train' else 'test')
            img_group = group.create_group('images')
            lbl_group = group.create_group('labels')

            # 1. Charger le JSON COCO
            json_file = os.path.join(in_dir, f"{split}.json")
            with open(json_file, 'r') as f:
                coco_data = json.load(f)

            # 2. Mapper les annotations aux images
            annots_map = {}
            for ann in coco_data['annotations']:
                img_id = ann['image_id']
                if img_id not in annots_map: annots_map[img_id] = []
                # COCO [x,y,w,h] -> format attendu par ton repo [class, x1, y1, x2, y2]
                x, y, w, h = ann['bbox']
                annots_map[img_id].append([ann['category_id'], x, y, x+w, y+h])

            # 3. Sauvegarder dans le H5
            for img_info in tqdm(coco_data['images'], desc=split):
                img_name = img_info['file_name']
                img_id = img_info['id']
                
                # Charger l'image JPG
                img_path = os.path.join(in_dir, split, img_name)
                img = np.array(Image.open(img_path).convert('L'))
                
                # Récupérer les labels (ou un tableau vide si aucun objet)
                labels = np.array(annots_map.get(img_id, []), dtype=np.float32)
                
                # Utiliser le nom sans extension comme clé
                key = os.path.splitext(img_name)[0]
                img_group.create_dataset(key, data=img)
                lbl_group.create_dataset(key, data=labels)

In [6]:
# On pointe vers le dossier où se trouvent train.json, val.json et les dossiers d'images
COCO_DIR = 'multiOrg_benchmark/multiorg_mmdet_coco'
OUTPUT_H5 = 'data/multiorg.h5'

if __name__ == "__main__":
    if not os.path.exists('data'): os.makedirs('data')
    
    print("Converting COCO crops to H5 format...")
    prepare_multiorg_coco(COCO_DIR, OUTPUT_H5)
    print(f"Done! {OUTPUT_H5} is ready for DINOv2.")

Converting COCO crops to H5 format...
Converting train to H5...


train: 100%|██████████| 20011/20011 [00:47<00:00, 423.49it/s]


Converting val to H5...


val: 100%|██████████| 2280/2280 [00:05<00:00, 427.36it/s]

Done! data/multiorg.h5 is ready for DINOv2.
